# 01 — Exploratory Data Analysis: CEMADEN Precipitation Trends

This notebook provides a reproducible EDA of the monthly precipitation dataset produced by the `src/cemaden_monthly_tolerance.py` ETL pipeline.

**Dataset origin**: CEMADEN sub-daily gauge records aggregated into monthly totals.  
**Tolerance rule**: A month is only included when ≥ 90 % of daily records are present AND missing days ≤ 3.

**Sections**
1. Load & inspect processed data
2. Data availability heatmap (Years × Months)
3. Long-term monthly hyetograph
4. Annual precipitation time-series and Mann-Kendall trend test

## 1. Imports & Configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import pymannkendall as mk

sns.set_theme(style="whitegrid", palette="muted")

DATA_DIR = Path("..") / "data" / "processed"
# Use geo-enriched file if available, fall back to merged
GEO_FILE = DATA_DIR / "df_monthly_filtered_with_geo_real.csv"
FALLBACK = DATA_DIR / "df_monthly_filtered_merged.csv"
DATA_FILE = GEO_FILE if GEO_FILE.exists() else FALLBACK

MONTH_NAMES = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]

print(f"Loading: {DATA_FILE.resolve()}")

## 2. Load & Inspect Data

In [ ]:
df = pd.read_csv(DATA_FILE)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
print("Stations per state:")
print(df.groupby("state")["gauge_code"].nunique().sort_values(ascending=False))
print(f"\nYears covered: {df['year'].min()} – {df['year'].max()}")
print(f"Total unique stations: {df['gauge_code'].nunique()}")

## 3. Select Station for Deep-Dive

Change `STATION` below to explore any gauge.

In [ ]:
# Pick the station with the most records as default
STATION = df.groupby("gauge_code")["rain_mm"].count().idxmax()
print(f"Selected station: {STATION}")
df_s = df[df["gauge_code"] == STATION].copy()

## 4. Data Availability Heatmap

Each cell shows whether a month-year combination has a valid rainfall record.  
Gaps (white cells) indicate months that failed the tolerance filter.

In [ ]:
pivot = df_s.pivot_table(index="year", columns="month", values="rain_mm", aggfunc="count")
pivot = pivot.reindex(columns=range(1, 13))
pivot.columns = MONTH_NAMES
pivot = pivot.sort_index(ascending=False)

fig, ax = plt.subplots(figsize=(12, max(4, len(pivot) * 0.35)))
sns.heatmap(
    pivot,
    ax=ax,
    cmap="YlGnBu",
    linewidths=0.5,
    linecolor="white",
    annot=True,
    fmt=".0f",
    cbar_kws={"label": "Records"},
)
ax.set_title(f"Data Availability — {STATION}", fontsize=14)
ax.set_xlabel("Month")
ax.set_ylabel("Year")
plt.tight_layout()
plt.show()

## 5. Monthly Hyetograph (Long-Term Mean)

Average rainfall for each calendar month across all available years.  
The dashed red line marks the overall long-term monthly mean.

In [ ]:
monthly_mean = df_s.groupby("month")["rain_mm"].mean().reindex(range(1, 13))
ltm = monthly_mean.mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(MONTH_NAMES, monthly_mean.values, color="#4C72B0", label="Mean Monthly Rainfall")
ax.axhline(ltm, color="red", linestyle="--", label=f"LTM: {ltm:.1f} mm")
ax.set_xlabel("Month")
ax.set_ylabel("Rainfall (mm)")
ax.set_title(f"Monthly Hyetograph — {STATION}")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Annual Precipitation & Mann-Kendall Trend Test

We compute the **Mann-Kendall** monotonic trend test (non-parametric, rank-based) and **Sen's slope** estimator using the `pymannkendall` library — a full Python replacement for the custom `ktaub.m` MATLAB script.

| Term | Meaning |
|---|---|
| **Trend** | `increasing`, `decreasing`, or `no trend` |
| **p-value** | Probability under H₀ (no trend); < 0.05 is statistically significant |
| **Sen's slope** | Median rate of change (mm per year) |
| **Tau (τ)** | Kendall's correlation coefficient in [−1, 1] |

In [ ]:
annual = df_s.groupby("year")["rain_mm"].sum().reset_index(name="annual_rain_mm")

# Linear trend line
coeffs = np.polyfit(annual["year"], annual["annual_rain_mm"], 1)
trend_y = np.polyval(coeffs, annual["year"])

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(annual["year"], annual["annual_rain_mm"], marker="o", label="Annual Rainfall")
ax.plot(annual["year"], trend_y, linestyle="--", color="red",
        label=f"OLS trend ({coeffs[0]:+.1f} mm/yr)")
ax.set_xlabel("Year")
ax.set_ylabel("Annual Rainfall (mm)")
ax.set_title(f"Annual Precipitation Trend — {STATION}")
ax.legend()
plt.tight_layout()
plt.show()

# Mann-Kendall test
if len(annual) >= 4:
    result = mk.original_test(annual["annual_rain_mm"])
    print("\n── Mann-Kendall Test Results ──")
    print(f"  Trend    : {result.trend}")
    print(f"  p-value  : {result.p:.4f}")
    print(f"  Sen slope: {result.slope:.2f} mm/yr")
    print(f"  Tau (τ)  : {result.Tau:.3f}")
    print(f"  z-stat   : {result.z:.3f}")
else:
    print("Insufficient data for Mann-Kendall test (need ≥ 4 years).")

## 7. Interactive Plotly Chart (Optional)

Hover over data points to inspect individual years.

In [ ]:
fig = px.line(
    annual, x="year", y="annual_rain_mm", markers=True,
    title=f"Annual Rainfall — {STATION}",
    labels={"annual_rain_mm": "Annual Rain (mm)", "year": "Year"},
)
fig.add_scatter(
    x=annual["year"], y=trend_y, mode="lines",
    name=f"Trend ({coeffs[0]:+.1f} mm/yr)",
    line=dict(dash="dash", color="red"),
)
fig.show()